In [1]:
!apt-get update && apt-get install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh

!pkill ollama

!pip install -q crewai crewai-tools playwright pydantic pandas sentence-transformers pypdf faiss-cpu nest_asyncio
!playwright install chromium

!apt-get update -y
!apt-get install -y \
    libatk1.0-0 \
    libatk-bridge2.0-0 \
    libcups2 \
    libdrm2 \
    libxkbcommon0 \
    libxcomposite1 \
    libxdamage1 \
    libxfixes3 \
    libxrandr2 \
    libgbm1 \
    libasound2 \
    libpangocairo-1.0-0 \
    libpango-1.0-0 \
    libgtk-3-0 \
    libnss3 \
    libnspr4 \
    libx11-xcb1 \
    libxcb1 \
    libxext6 \
    libx11-6

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [93.4 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.2 MB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,004 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,915 kB]


In [2]:
import re
import pandas as pd
import numpy as np
from typing import List, Optional, Dict
import time
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import BaseTool


import subprocess

# Use the ABSOLUTE PATH to prevent 'FileNotFoundError'
try:
    subprocess.Popen(
        ["/usr/local/bin/ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    print("✅ Ollama server is starting...")
except FileNotFoundError:
    print("❌ Error: Ollama not found. Re-running installer...")
    !curl -fsSL https://ollama.com/install.sh | sh

# Give the GPU 15 seconds to map memory
time.sleep(15)

# 5. Pull the 8B model (This will be 10x faster now)
print("Pulling Llama 3.1 8B...")
!/usr/local/bin/ollama pull llama3.1:8b

print("\n🚀 GPU is engaged and Llama-3.1-8B is ready for 281,880 rows!")

✅ Ollama server is starting...
Pulling Llama 3.1 8B...


🚀 GPU is engaged and Llama-3.1-8B is ready for 281,880 rows!


In [3]:
from pydantic import BaseModel, Field
from typing import Optional, List

class Product(BaseModel):
    company: Optional[str] = Field(
        default=None,
        description="Name of the company or brand of the product"
    )
    product: Optional[str] = Field(
        default=None,
        description="Category or type of the product (e.g., smartphone, laptop)"
    )
    model: Optional[str] = Field(
        default=None,
        description="Specific model name or number of the product"
    )
    color: Optional[str] = Field(
        default=None,
        description="Color of the product"
    )
    price: Optional[float] = Field(
        default=None,
        ge=0,
        description="Price of the product (must be non-negative)"
    )
    location: Optional[str] = Field(
        default=None,
        description="Location where the product is being sold"
    )
    platform: Optional[str] = Field(
        default=None,
        description="Platform or marketplace where the product is listed (e.g., Amazon, Noon)"
    )


class ProductList(BaseModel):
    products: List[Product]

In [4]:
import asyncio
import re
import ast
from playwright.async_api import async_playwright
from crewai.tools import tool
import nest_asyncio
nest_asyncio.apply()

WEBSITE_URLS = {
    "amazon": "https://www.amazon.ae/s?k=smartphones",
    "noon": "https://www.noon.com/uae-en/electronics-and-mobiles/mobiles-20905/",
    "sharafdg": "https://uae.sharafdg.com/c/mobiles-tablets/smartphones/",
    "ebay": "https://www.ebay.com/sch/i.html?_nkw=smartphones",
    "emax": "https://www.emaxme.com/en/mobiles-tablets/mobiles/smartphones"
}

def clean_price(price):
    if not price: return None
    price = re.sub(r"[^\d.]", "", price)
    try:
        return float(price)
    except:
        return None

def normalize_sites(sites):
    if isinstance(sites, str):
        try:
            sites = ast.literal_eval(sites)
        except:
            sites = [sites]
    return sites if isinstance(sites, list) else [sites]


@tool("Smartphone Scraper Tool")
async def smartphone_scraper_tool(sites):
    """
    Scrapes smartphones from ecommerce sites using Playwright.
    """
    sites = normalize_sites(sites)
    results = []

    async with async_playwright() as p:
        # 1. Launch with more 'human-like' flags
        browser = await p.chromium.launch(
            headless=True,
            args=[
                "--no-sandbox",
                "--disable-dev-shm-usage",
                "--disable-blink-features=AutomationControlled" # Helps bypass bot detection
            ]
        )

        for site in sites:
            site_key = str(site).lower().strip()
            if site_key not in WEBSITE_URLS: continue

            # 2. Create a NEW context and page for every single site
            context = await browser.new_context(
                user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
                viewport={'width': 1920, 'height': 1080}
            )
            page = await context.new_page()

            try:
                print(f"Navigating to {site_key}...")
                # 3. Use 'load' instead of 'domcontentloaded' for steadier connections on Noon/SharafDG
                await page.goto(WEBSITE_URLS[site_key], timeout=60000, wait_until="load")

                # Human-like pause
                await asyncio.sleep(3)

                # Scroll slowly
                await page.mouse.wheel(0, 10000)
                await asyncio.sleep(2)

                cards = await page.query_selector_all(".s-result-item, .productContainer, .s-item, .product-item, [data-qa='product-card']")

                for card in cards[:80]:
                    title_el = await card.query_selector("h2, .product-title, .s-item__title, .name, [data-qa='product-name']")
                    price_el = await card.query_selector(".a-price-whole, .amount, .s-item__price, .price, [data-qa='product-price']")

                    if title_el:
                        results.append({
                            "title": (await title_el.inner_text()).strip(),
                            "price": clean_price(await price_el.inner_text()) if price_el else None,
                            "platform": site_key
                        })

                print(f"Successfully scraped {len(results)} items from {site_key}")

            except Exception as e:
                print(f"Error on {site_key}: {str(e)[:100]}") # Print first 100 chars of error

            finally:
                # 4. Always close the context before moving to the next site
                await context.close()

        await browser.close()
    return results

In [5]:
llm_model = LLM(
    model="ollama/llama3.1:8B",
    base_url="http://localhost:11434",
    temperature=0,
    max_tokens=500
)

In [6]:
from crewai import Agent


# 1. Product Scraper: Needs the tool and must handle the async result
scraping_agent = Agent(
    role="Product Scraper",
    goal="Extract raw smartphone listings from {sites} using the scraper tool",
    backstory=(
        "You are a technical expert in web navigation. You know how to use "
        "the Smartphone Scraper Tool to gather raw data from multiple "
        "ecommerce platforms. You focus on getting raw, unformatted text and prices."
    ),
    tools=[smartphone_scraper_tool],
    llm=llm_model,
    verbose=True,
    allow_delegation=False  # Keep it focused on the technical task
)

# 2. Data Structurer: The bridge between raw text and your Pydantic model
structuring_agent = Agent(
    role="Data Structurer",
    goal="Parse raw scraped text into a preliminary structured format",
    backstory=(
        "You are an expert at identifying patterns in messy data. You take "
        "raw titles like 'iPhone 15 Pro 128GB Blue' and extract the brand (Apple), "
        "model (iPhone 15 Pro), and color (Blue)."
    ),
    llm=llm_model,
    verbose=True
)

# 3. Data Validator: The final quality gate using your Pydantic Schema
validator_agent = Agent(
    role="Data Validator",
    goal="Refine, deduplicate, and finalize the smartphone dataset",
    backstory=(
        "You are a data quality perfectionist. You ensure that prices are "
        "accurate floats, remove duplicate listings from different sources, "
        "and ensure the final output matches the required JSON schema perfectly."
    ),
    llm=llm_model,
    verbose=True
)

In [7]:
from crewai import Task

# 1. Scraping Task: Focus on Tool Execution
scraping_task = Task(
    description="""
    Identify and extract smartphone listings from these specific platforms: {sites}.
    Use the 'Smartphone Scraper Tool' to gather the raw title and price text.
    Do not worry about formatting yet; just ensure you capture as many relevant
    smartphone entries as possible.
    """,
    expected_output="A raw list of smartphone titles and prices for each platform.",
    agent=scraping_agent,
    tools=[smartphone_scraper_tool]
)

# 2. Structuring Task: Focus on Parsing and Categorization
structuring_task = Task(
    description="""
    Using the raw data from the scraper, break down each entry into a structured format.
    You must identify:
    - 'company' (The brand, e.g., Apple, Samsung)
    - 'model' (e.g., iPhone 15, Galaxy S24)
    - 'color' (if mentioned in the title)
    - 'price' (Cleaned numeric value)
    - 'platform' (Amazon, Noon, etc.)
    - 'location' (Based on the platform region, e.g., UAE or Global)

    Set 'product' to 'Smartphone' for every entry.
    """,
    expected_output="A preliminary JSON list containing mapped product details.",
    agent=structuring_agent,
    context=[scraping_task]
)

# 3. Validation Task: Focus on Final Schema and Quality
validation_task = Task(
    description="""
    Perform a final quality audit on the structured data:
    1. Remove any duplicate listings (check for same model and price).
    2. Ensure 'price' is a strictly positive float or null if unavailable.
    3. Remove entries that are missing both model and company names.
    4. Ensure the final output matches the ProductList schema exactly.
    """,
    expected_output="A final, cleaned, and validated list of smartphones in structured JSON.",
    agent=validator_agent,
    context=[structuring_task],
    output_json=ProductList  # This enforces your Pydantic model on the final result
)

In [8]:
from crewai import Crew

crew = Crew(
    agents=[scraping_agent, structuring_agent, validator_agent],
    tasks=[scraping_task, structuring_task, validation_task],
    verbose=True
)


In [ ]:
result = crew.kickoff(inputs={
    "sites": ["amazon", "noon", "sharafdg", "ebay", "emax"]
})

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: f649138c-d900-45e1-b4ce-a6f6db5eaa95                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Identify and extract smartphone listings from these specific platforms: ['amazon', 'noon', 'sharafdg',     │
│  'ebay', 'emax'].                                                                                               │
│      Use the 'Smartphone Scraper Tool' to gather the raw title and price text.                                  │
│      Do not worry about formatting yet; just ensure you capture as many relevant                                │
│      smartphone entries as possible.                                                                            │
│                                                                                                                 │
│  ID: 7e8871e3-e7b5-448b-a0bd-e54856694572                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Product Scraper                                                                                         │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Identify and extract smartphone listings from these specific platforms: ['amazon', 'noon', 'sharafdg',     │
│  'ebay', 'emax'].                                                                                               │
│      Use the 'Smartphone Scraper Tool' to gather the raw title and price text.                                  │
│      Do not worry about formatting yet; just ensure you capture as many relevant                                │
│      smartphone entries as possible.                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: smartphone_scraper_tool                                                                                  │
│  Args: {'sites': ['amazon', 'noon', 'sharafdg', 'ebay', 'emax']}                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Navigating to amazon...
Successfully scraped 13 items from amazon
Navigating to noon...
Error on noon: Page.goto: net::ERR_HTTP2_PROTOCOL_ERROR at https://www.noon.com/uae-en/electronics-and-mobiles/mobi
Navigating to sharafdg...
Successfully scraped 13 items from sharafdg
Navigating to ebay...
Successfully scraped 13 items from ebay
Navigating to emax...
Successfully scraped 13 items from emax
Tool smartphone_scraper_tool executed with result: [{'title': 'Results', 'price': None, 'platform': 'amazon'}, {'title': 'Samsung Galaxy A07 LTE (International Version) Smartphone, 128GB Storage, 4GB RAM, 6nm Processor, Large Display, 6x OS Upgrades, ...


╭─────────────────────────────────────── ✅ Tool Execution Completed (#1) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: smartphone_scraper_tool                                                                                  │
│  Output: [{'title': 'Results', 'price': None, 'platform': 'amazon'}, {'title': 'Samsung Galaxy A07 LTE          │
│  (International Version) Smartphone, 128GB Storage, 4GB RAM, 6nm Processor, Large Display, 6x OS Upgrades,      │
│  Black', 'price': 387.0, 'platform': 'amazon'}, {'title': 'Samsung Galaxy A06 LTE, Android Smartphone, Dual     │
│  SIM Mobile Phone, 4GB RAM, 64GB Storage, Black (UAE Version)', 'price': 324.0, 'platform': 'amazon'},          │
│  {'title': 'Motorola Moto E15 Dual-SIM 64GB ROM + 2GB RAM (GSM Only | No CDMA) Factory Unlocked 4G/LTE          │
│  Smartphone (Fresh Lavender) - International Version', 'price': 287.0, 'platform': 'amazon'}, {'title': 'More   │
│  results', 'price': None, 'platform': 'amazon'}, {'title': 'Samsung Galaxy A17 LTE, Android Smartphone, 128GB   │
│  Storage, 6GB RAM, Black, 6x OS Upgrades, Large Display, 50MP OIS Camera (UAE Version)', 'price': 599.0,        │
│  'platform': 'amazon'}, {'title': 'Samsung Galaxy A06 LTE, Android Smartphone, Dual SIM Mobile Phone, 4GB RAM,  │
│  64GB Storage, Light Blue (UAE Version)', 'price': 324.0, 'platform': 'amazon'}, {'title': 'Samsung Galaxy A17  │
│  5G Blue (8 GB RAM / 256\u202fGB Storage) Android Smartphone | 6x OS Upgrades, Large Display, 50MP OIS Camera   │
│  | International Version', 'price': 728.0, 'platform': 'amazon'}, {'title': 'HONOR X5c Plus 4G Smartphone, 4GB  │
│  RAM 128GB ROM, Dual SIM, 5260mAh Battery, 6.74" 90Hz Display, 50MP Camera, Ocean Cyan – Middle East Version    │
│  (UAE Type Approved)', 'price': 359.0, 'platform': 'amazon'}, {'title': 'HONOR X6c Smartphone 6GB RAM 256GB     │
│  ROM, 6.61" 120Hz Display, 50MP Camera, 35W Fast Charging, Dual SIM, Moonlight White – Middle East Version      │
│  (UAE Type Approved)', 'price': 499.0, 'platform': 'amazon'}, {'title': 'Redmi A5 Midnight Black 4GB RAM 128GB  │
│  ROM', 'price': 348.0, 'platform': 'amazon'}, {'title': 'Redmi Note 15 Smartphone (8GB RAM, 256GB Storage) –    │
│  6000mAh Battery, 108MP Camera, 6.77" FHD+ Display, IP64 Water & Dust Resistant, Black – UAE Warranty',         │
│  'price': 685.0, 'platform': 'amazon'}, {'title': 'Redmi Note 15 Smartphone (8GB RAM, 256GB Storage) – 6000mAh  │
│  Battery, 108MP Camera, 6.77" FHD+ Display, IP64 Water & Dust Resistant, Glacier Blue – UAE Warranty',          │
│  'price': 715.0, 'platform': 'amazon'}]                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Product Scraper                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {"name": "analyze_tool_result", "parameters": {"tool_output": "[{'title': 'Results', 'price': None,            │
│  'platform': 'amazon'}, {'title': 'Samsung Galaxy A07 LTE (International Version) Smartphone, 128GB Storage,    │
│  4GB RAM, 6nm Processor, Large Display, 6x OS Upgrades, Black', 'price': 387.0, 'platform': 'amazon'}]"}}       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      Identify and extract smartphone listings from these specific platforms: ['amazon', 'noon', 'sharafdg',     │
│  'ebay', 'emax'].                                                                                               │
│      Use the 'Smartphone Scraper Tool' to gather the raw title and price text.                                  │
│      Do not worry about formatting yet; just ensure you capture as many relevant                                │
│      smartphone entries as possible.                                                                            │
│                                                                                                                 │
│  Agent: Product Scraper                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Using the raw data from the scraper, break down each entry into a structured format.                       │
│      You must identify:                                                                                         │
│      - 'company' (The brand, e.g., Apple, Samsung)                                                              │
│      - 'model' (e.g., iPhone 15, Galaxy S24)                                                                    │
│      - 'color' (if mentioned in the title)                                                                      │
│      - 'price' (Cleaned numeric value)                                                                          │
│      - 'platform' (Amazon, Noon, etc.)                                                                          │
│      - 'location' (Based on the platform region, e.g., UAE or Global)                                           │
│                                                                                                                 │
│      Set 'product' to 'Smartphone' for every entry.                                                             │
│                                                                                                                 │
│  ID: 95b3a45f-360d-4aa5-898d-905a1524573b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Data Structurer                                                                                         │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Using the raw data from the scraper, break down each entry into a structured format.                       │
│      You must identify:                                                                                         │
│      - 'company' (The brand, e.g., Apple, Samsung)                                                              │
│      - 'model' (e.g., iPhone 15, Galaxy S24)                                                                    │
│      - 'color' (if mentioned in the title)                                                                      │
│      - 'price' (Cleaned numeric value)                                                                          │
│      - 'platform' (Amazon, Noon, etc.)                                                                          │
│      - 'location' (Based on the platform region, e.g., UAE or Global)                                           │
│                                                                                                                 │
│      Set 'product' to 'Smartphone' for every entry.                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
import pandas as pd
import json

data = json.loads(result) if isinstance(result, str) else result

df = pd.DataFrame(data)

df.drop_duplicates(subset=["model", "price"], inplace=True)

df.to_csv("smartphones.csv", index=False)

print("✅ CSV saved: smartphones.csv")